# LlamaIndex Tutorial 2: RAG with Pinecone Vector Store

This notebook demonstrates how to use Pinecone cloud vector database with LlamaIndex.

## What you'll learn:
- Setting up Pinecone cloud vector database
- Indexing documents with Pinecone
- Querying with metadata filtering
- Production-ready vector storage

In [ ]:
# Install required packages
!pip install llama-index llama-index-llms-groq llama-index-embeddings-huggingface llama-index-vector-stores-pinecone pinecone-client python-dotenv

In [1]:
import os
from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

if not groq_api_key:
    raise ValueError("Please set GROQ_API_KEY in your .env file")
if not pinecone_api_key:
    raise ValueError("Please set PINECONE_API_KEY in your .env file")

print("✅ Environment loaded successfully!")

/Users/rushikeshvinodkharche/opt/anaconda3/envs/meet/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/Users/rushikeshvinodkharche/opt/anaconda3/envs/meet/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Environment loaded successfully!


In [2]:
# Initialize Groq LLM
llm = Groq(model="llama-3.3-70b-versatile", temperature=0.7, api_key=groq_api_key)

# Initialize HuggingFace Embedding (BGE-small, 384 dimensions)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", device="cpu")

Settings.llm = llm
Settings.embed_model = embed_model

print("✅ LLM and Embedding model initialized!")

✅ LLM and Embedding model initialized!


## Step 2: Set Up Pinecone

In [4]:
# Initialize Pinecone
pc = Pinecone(api_key=pinecone_api_key)

index_name = "llamaindex-rag-tutorial"

# Check if index exists in your current indexes
existing_indexes = [idx['name'] for idx in pc.list_indexes()]

if index_name in existing_indexes:
    print(f"✅ Index {index_name} already exists")
else:
    # You need to delete an old index first or use namespaces
    print(f"❌ Cannot create index. Current indexes ({len(existing_indexes)}/5):")
    for idx in existing_indexes:
        print(f"  - {idx}")
    print("\nOptions:")
    print("1. Delete an unused index: pc.delete_index('index-name')")
    print("2. Use namespaces in an existing index")
    print("3. Upgrade your Pinecone plan")

pinecone_index = pc.Index(index_name)

✅ Index llamaindex-rag-tutorial already exists


## Step 3: Create Documents and Index

In [5]:
documents = [
    Document(text="Machine learning is a subset of artificial intelligence that enables systems to learn from data.", metadata={"category": "AI", "year": 2023}),
    Document(text="Deep learning uses neural networks with multiple layers to model complex patterns.", metadata={"category": "AI", "year": 2023}),
    Document(text="Natural language processing helps computers understand and generate human language.", metadata={"category": "NLP", "year": 2024}),
    Document(text="Vector databases store embeddings for fast similarity search in RAG systems.", metadata={"category": "Database", "year": 2024}),
]
print(f"✅ Created {len(documents)} documents")

✅ Created 4 documents


In [6]:
# Create Pinecone vector store
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Create index
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)
print("✅ Index created in Pinecone!")

Upserted vectors: 100%|██████████| 4/4 [00:03<00:00,  1.22it/s]

✅ Index created in Pinecone!


## Step 4: Query with Metadata Filtering

In [7]:
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterOperator

# Query without filter
query_engine = index.as_query_engine()
response = query_engine.query("What is machine learning?")
print(f"Query: What is machine learning?\n")
print(f"Response: {response}\n")

Query: What is machine learning?

Response: Machine learning is a subset of artificial intelligence that enables systems to learn from data.



In [8]:
# Query with metadata filter
filters = MetadataFilters(
    filters=[MetadataFilter(key="category", operator=FilterOperator.EQ, value="AI")]
)

query_engine_filtered = index.as_query_engine(filters=filters)
response = query_engine_filtered.query("Explain deep learning")
print(f"Query (filtered by category=AI): Explain deep learning\n")
print(f"Response: {response}\n")

Query (filtered by category=AI): Explain deep learning

Response: Deep learning uses neural networks with multiple layers to model complex patterns.



## Summary

In this tutorial, we learned:
1. ✅ How to set up Pinecone cloud vector database
2. ✅ How to index documents with Pinecone
3. ✅ How to query with metadata filtering
4. ✅ Production-ready vector storage setup

### Key Takeaways:
- **Pinecone** is a managed cloud vector database
- Supports metadata filtering for structured queries
- Production-ready with high scalability
- Perfect for production RAG applications

### Next Steps:
- Try different metadata filters
- Experiment with hybrid search (Tutorial 5)
- Move to Tutorial 3 to learn about Weaviate